# Text-to-SQL QLoRA training (Colab)

Thin wrapper around `src/train.py` — all real logic lives in the repo, not here, so Colab/Kaggle stay identical. Runs 3 ablation ranks (r=8/16/32) then the final full-budget run with the winning rank.

**Checkpoints save to Google Drive** so a ~90-min idle disconnect loses at most `save_steps` worth of progress — re-running the same cell resumes automatically.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
CHECKPOINT_ROOT = '/content/drive/MyDrive/text-to-sql-qlora/checkpoints'
os.makedirs(CHECKPOINT_ROOT, exist_ok=True)
# Export to the OS environment, not just the Python namespace — "!" shell
# cells below resolve $CHECKPOINT_ROOT from os.environ, not from this variable.
os.environ['CHECKPOINT_ROOT'] = CHECKPOINT_ROOT

In [ ]:
!git clone https://github.com/Rick-Roy-JC/-text-to-sql-qlora.git
%cd -text-to-sql-qlora
!pip install -q -r requirements.txt

## Ablation runs: r=8, r=16, r=32 (2,500-example subset, 2 epochs each, ~25-30 min/run on T4)

In [ ]:
!python src/train.py --regime ablation --lora-rank 8 --output-root "$CHECKPOINT_ROOT"

In [ ]:
!python src/train.py --regime ablation --lora-rank 16 --output-root "$CHECKPOINT_ROOT"

In [ ]:
!python src/train.py --regime ablation --lora-rank 32 --output-root "$CHECKPOINT_ROOT"

## Compare ablation results

Inspect `results/train_metrics_ablation_r*.json` (eval loss on val split) before picking the winning rank for the final run. Full exact-match/execution-accuracy comparison happens in Phase 4 (`src/eval.py`), not here.

In [ ]:
import json, glob
for path in sorted(glob.glob('results/train_metrics_ablation_r*.json')):
    print(path, json.load(open(path)))

## Final run: winning rank, full ~6,294-example train split, 3 epochs

Set `WINNING_RANK` based on the ablation comparison above.

In [ ]:
WINNING_RANK = 16  # update after inspecting ablation results
import os
os.environ['WINNING_RANK'] = str(WINNING_RANK)
!python src/train.py --regime final --lora-rank $WINNING_RANK --output-root "$CHECKPOINT_ROOT"